# Topic Classifier — Final Clean Training
### DistilBERT · L1 (Vertical) + L2 (Subdomain)
Run cells top to bottom. Uses `dataset_final.csv`

## Cell 1 — Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset from Drive to Colab working directory
import shutil
shutil.copy('/content/drive/MyDrive/dataset_final.csv', '/content/dataset_final.csv')
print("✅ Dataset copied")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset copied


In [ ]:
import sys
!{sys.executable} -m pip install transformers datasets scikit-learn pandas torch accelerate -q
print('✅ Done')

✅ Done


## Cell 2 — Imports & Config

In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection  import GroupShuffleSplit
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import accuracy_score, f1_score, classification_report
from torch.utils.data         import Dataset
from transformers             import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
warnings.filterwarnings('ignore')

DATASET_PATH = 'dataset_final.csv'
MODEL_L1_DIR = 'model_level1'
MODEL_L2_DIR = 'model_level2'
ENCODER_DIR  = 'label_encoders'

BASE_MODEL  = 'distilbert-base-uncased'
MAX_LEN     = 64
BATCH_SIZE  = 32
EPOCHS      = 6
LR          = 2e-5
SEED        = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU — training will be slow. Use Colab T4 GPU for speed.')

os.makedirs(MODEL_L1_DIR, exist_ok=True)
os.makedirs(MODEL_L2_DIR, exist_ok=True)
os.makedirs(ENCODER_DIR,  exist_ok=True)
print('✅ Config ready')

Device : cuda
GPU    : Tesla T4
✅ Config ready


## Cell 3 — Load & Inspect Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
df.dropna(subset=['text','label_vertical','label_subdomain'], inplace=True)
df['text']            = df['text'].astype(str).str.strip()
df['label_vertical']  = df['label_vertical'].str.strip()
df['label_subdomain'] = df['label_subdomain'].str.strip()
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f'Total rows   : {len(df)}')
print(f'L1 classes   : {df["label_vertical"].nunique()}')
print(f'L2 classes   : {df["label_subdomain"].nunique()}')
print()
print('L1 distribution:')
print(df['label_vertical'].value_counts().to_string())
print()
df.head(3)

Total rows   : 2897
L1 classes   : 5
L2 classes   : 41

L1 distribution:
label_vertical
Technology    1021
Politics       845
Finance        574
Sports         257
General        200



,text,label_vertical,label_subdomain
0,The Prime Minister chaired a cabinet meeting t...,Politics,Indian Politics
1,The Home Minister discussed internal security ...,Politics,Indian Politics
2,The Finance Minister presented key highlights ...,Politics,Indian Politics


## Cell 4 — L2 Distribution Check

In [ ]:
l2 = df['label_subdomain'].value_counts()
print(f'L2 max : {l2.idxmax()} = {l2.max()}')
print(f'L2 min : {l2.idxmin()} = {l2.min()}')
print(f'Imbalance ratio: {l2.max()/l2.min():.1f}x')
print()
print(l2.to_string())

L2 max : Artificial Intelligence = 185
L2 min : Athletics & Olympics = 10
Imbalance ratio: 18.5x

label_subdomain
Artificial Intelligence           185
Cricket                           115
USA Politics                      111
Stock Market                      110
UK Politics                       109
Cryptocurrency                    109
Geopolitics                       107
Cybersecurity                     104
International Relations           100
Banking & Loans                   100
Elections & Voting                 99
Space & Exploration                98
Defence & Military                 97
Smartphones & Gadgets              96
Telecom & Networks                 95
Indian Politics                    94
Software & Apps                    94
Social Media & Internet            93
Electric Vehicles & Clean Tech     93
Semiconductors & Chips             91
Political Parties                  88
Big Tech Companies                 79
Personal Finance                   55
Football    

## Cell 5 — Tokenizer & Dataset Class

In [ ]:
print('Loading tokenizer...')
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_MODEL)

class TopicDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True,
            padding='max_length', max_length=MAX_LEN
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy' : accuracy_score(labels, preds),
        'f1_macro' : f1_score(labels, preds, average='macro', zero_division=0)
    }

print('✅ Tokenizer & Dataset class ready')

Loading tokenizer...
✅ Tokenizer & Dataset class ready


## Cell 6 — Training Function (with weighted loss + group split)

In [ ]:
def train_and_evaluate(texts, raw_labels, output_dir, encoder_path, level_name):

    print(f'\n{"="*60}')
    print(f'  Training {level_name}')
    print(f'{"="*60}')

    le = LabelEncoder()
    encoded = le.fit_transform(raw_labels)
    num_classes = len(le.classes_)
    print(f'Classes ({num_classes}): {list(le.classes_)}')

    with open(encoder_path, 'wb') as f:
        pickle.dump(le, f)
    print(f'Encoder saved → {encoder_path}')

    texts_arr   = np.array(texts)
    encoded_arr = np.array(encoded)

    # GroupShuffleSplit — near-duplicates stay in same split, no leakage
    groups = np.array([' '.join(t.lower().split()[:8]) for t in texts])

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
    train_val_idx, test_idx = next(gss1.split(texts_arr, encoded_arr, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.111, random_state=SEED)
    tr_idx, val_idx = next(gss2.split(
        texts_arr[train_val_idx],
        encoded_arr[train_val_idx],
        groups=groups[train_val_idx]
    ))
    train_idx = train_val_idx[tr_idx]
    val_idx   = train_val_idx[val_idx]

    X_tr  = texts_arr[train_idx].tolist()
    X_val = texts_arr[val_idx].tolist()
    X_te  = texts_arr[test_idx].tolist()
    y_tr  = encoded_arr[train_idx].tolist()
    y_val = encoded_arr[val_idx].tolist()
    y_te  = encoded_arr[test_idx].tolist()

    print(f'Train={len(X_tr)}  Val={len(X_val)}  Test={len(X_te)}')

    missing = set(encoded) - set(y_tr)
    if missing:
        print(f'⚠️  Classes missing from train: {[le.classes_[i] for i in missing]}')
    else:
        print(f'✅ All {num_classes} classes in train')

    train_ds = TopicDataset(X_tr,  y_tr)
    val_ds   = TopicDataset(X_val, y_val)
    test_ds  = TopicDataset(X_te,  y_te)

    model = DistilBertForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=num_classes
    )

    # Weighted loss — handles class imbalance
    class_counts = np.bincount(y_tr, minlength=num_classes).astype(float)
    class_counts = np.where(class_counts == 0, 1, class_counts)  # avoid div by zero
    raw_weights  = 1.0 / class_counts
    cw           = torch.tensor(raw_weights / raw_weights.sum() * num_classes,
                                dtype=torch.float)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.get('labels')
            outputs = model(**inputs)
            logits  = outputs.get('logits')
            loss    = nn.CrossEntropyLoss(weight=cw.to(logits.device))(logits, labels)
            return (loss, outputs) if return_outputs else loss

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        seed=SEED,
        logging_steps=50,
        report_to='none',
        fp16=torch.cuda.is_available()
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    out   = trainer.predict(test_ds)
    preds = np.argmax(out.predictions, axis=-1)
    y_str = le.inverse_transform(y_te)
    p_str = le.inverse_transform(preds)

    acc = accuracy_score(y_te, preds)
    f1  = f1_score(y_te, preds, average='macro', zero_division=0)

    print(f'\n── {level_name} — Test Results ──')
    print(f'Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'F1 Macro : {f1:.4f}')
    print('\nPer-class report:')
    print(classification_report(y_str, p_str, zero_division=0))

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'✅ Model saved → {output_dir}')

    return le, acc, f1

print('✅ Training function ready')

✅ Training function ready


## Cell 7 — Train Level 1 (Vertical)
Predicts: Politics / Sports / Technology / Finance / General

⏱️ ~2 min on GPU, ~20 min on CPU

In [ ]:
texts = df['text'].tolist()

le_l1, acc_l1, f1_l1 = train_and_evaluate(
    texts        = texts,
    raw_labels   = df['label_vertical'].tolist(),
    output_dir   = MODEL_L1_DIR,
    encoder_path = f'{ENCODER_DIR}/le_level1.pkl',
    level_name   = 'LEVEL 1 — Vertical Classifier'
)

print(f'\n🎯 L1 Accuracy: {acc_l1*100:.2f}%  |  F1: {f1_l1:.4f}')


  Training LEVEL 1 — Vertical Classifier
Classes (5): [np.str_('Finance'), np.str_('General'), np.str_('Politics'), np.str_('Sports'), np.str_('Technology')]
Encoder saved → label_encoders/le_level1.pkl
Train=2317  Val=290  Test=290
✅ All 5 classes in train


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.420991,0.266724,0.962069,0.945120
2,0.342984,0.048125,0.982759,0.982507
3,0.035063,0.035520,0.982759,0.977751
4,0.022593,0.032562,0.989655,0.987668
5,0.012500,0.035684,0.989655,0.987666
6,0.010360,0.035927,0.989655,0.987666


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



── LEVEL 1 — Vertical Classifier — Test Results ──
Accuracy : 0.9862  (98.62%)
F1 Macro : 0.9839

Per-class report:
              precision    recall  f1-score   support

     Finance       0.98      0.95      0.96        57
     General       0.94      1.00      0.97        17
    Politics       0.98      1.00      0.99        84
      Sports       1.00      1.00      1.00        18
  Technology       1.00      0.99      1.00       114

    accuracy                           0.99       290
   macro avg       0.98      0.99      0.98       290
weighted avg       0.99      0.99      0.99       290



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved → model_level1

🎯 L1 Accuracy: 98.62%  |  F1: 0.9839


## Cell 8 — Train Level 2 (Subdomain)
⏱️ ~5 min on GPU, ~40 min on CPU

In [ ]:
le_l2, acc_l2, f1_l2 = train_and_evaluate(
    texts        = texts,
    raw_labels   = df['label_subdomain'].tolist(),
    output_dir   = MODEL_L2_DIR,
    encoder_path = f'{ENCODER_DIR}/le_level2.pkl',
    level_name   = 'LEVEL 2 — Subdomain Classifier'
)

print(f'\n🎯 L2 Accuracy: {acc_l2*100:.2f}%  |  F1: {f1_l2:.4f}')


  Training LEVEL 2 — Subdomain Classifier
Classes (41): [np.str_('Acknowledgements'), np.str_('Artificial Intelligence'), np.str_('Athletics & Olympics'), np.str_('Badminton'), np.str_('Banking & Loans'), np.str_('Basketball'), np.str_('Big Tech Companies'), np.str_('Chitchat'), np.str_('Cricket'), np.str_('Cryptocurrency'), np.str_('Cybersecurity'), np.str_('Defence & Military'), np.str_('Elections & Voting'), np.str_('Electric Vehicles & Clean Tech'), np.str_('Football'), np.str_('Formula 1 & Motorsport'), np.str_('General Greetings'), np.str_('Geopolitics'), np.str_('Global Economy'), np.str_('Government Policy'), np.str_('Indian Politics'), np.str_('Insurance'), np.str_('International Relations'), np.str_('Interruptions'), np.str_('Kabaddi & Wrestling'), np.str_('Mutual Funds & SIP'), np.str_('Out of Scope'), np.str_('Personal Finance'), np.str_('Political Parties'), np.str_('Real Estate'), np.str_('Semiconductors & Chips'), np.str_('Smartphones & Gadgets'), np.str_('Social Media 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,3.678399,3.269970,0.379310,0.323198
2,3.267970,2.446367,0.768966,0.746889
3,2.204383,1.850430,0.865517,0.868008
4,1.829629,1.442747,0.910345,0.906009
5,1.347593,1.225843,0.941379,0.929996
6,1.222211,1.155942,0.944828,0.931460


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



── LEVEL 2 — Subdomain Classifier — Test Results ──
Accuracy : 0.9552  (95.52%)
F1 Macro : 0.9294

Per-class report:
                                precision    recall  f1-score   support

              Acknowledgements       0.50      1.00      0.67         2
       Artificial Intelligence       1.00      0.88      0.94        17
               Banking & Loans       0.92      0.92      0.92        12
            Big Tech Companies       1.00      1.00      1.00        11
                      Chitchat       1.00      1.00      1.00         8
                       Cricket       1.00      1.00      1.00         8
                Cryptocurrency       0.91      0.91      0.91        11
                 Cybersecurity       1.00      0.85      0.92        13
            Defence & Military       1.00      1.00      1.00        13
            Elections & Voting       1.00      1.00      1.00        11
Electric Vehicles & Clean Tech       1.00      1.00      1.00        10
                 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved → model_level2

🎯 L2 Accuracy: 95.52%  |  F1: 0.9294


## Cell 9 — Save Taxonomy Map

In [ ]:
taxonomy = (
    df.groupby('label_vertical')['label_subdomain']
    .unique().apply(sorted).to_dict()
)
with open(f'{ENCODER_DIR}/taxonomy.json', 'w') as f:
    json.dump(taxonomy, f, indent=2)
print('✅ Taxonomy saved')
print(json.dumps(taxonomy, indent=2))

✅ Taxonomy saved
{
  "Finance": [
    "Banking & Loans",
    "Cryptocurrency",
    "Global Economy",
    "Insurance",
    "Mutual Funds & SIP",
    "Personal Finance",
    "Real Estate",
    "Stock Market",
    "Taxation"
  ],
  "General": [
    "Acknowledgements",
    "Artificial Intelligence",
    "Chitchat",
    "General Greetings",
    "Interruptions",
    "Out of Scope"
  ],
  "Politics": [
    "Defence & Military",
    "Elections & Voting",
    "Geopolitics",
    "Government Policy",
    "Indian Politics",
    "International Relations",
    "Political Parties",
    "UK Politics",
    "USA Politics"
  ],
  "Sports": [
    "Athletics & Olympics",
    "Badminton",
    "Basketball",
    "Cricket",
    "Football",
    "Formula 1 & Motorsport",
    "Kabaddi & Wrestling",
    "Tennis"
  ],
  "Technology": [
    "Artificial Intelligence",
    "Big Tech Companies",
    "Cybersecurity",
    "Electric Vehicles & Clean Tech",
    "Semiconductors & Chips",
    "Smartphones & Gadgets",
    "So

## Cell 10 — Final Summary

In [ ]:
print('='*60)
print('  TRAINING COMPLETE')
print('='*60)
print(f'  L1 Vertical  Accuracy : {acc_l1*100:.2f}%  |  F1: {f1_l1:.4f}')
print(f'  L2 Subdomain Accuracy : {acc_l2*100:.2f}%  |  F1: {f1_l2:.4f}')
print('='*60)
print('\nExpected honest ranges:')
print('  L1: 88-94%   L2: 78-87%')
print('  If you see 99%+ something is wrong (leakage)')
print('\nFolders to copy into Streamlit project:')
print('  model_level1/   model_level2/   label_encoders/')

  TRAINING COMPLETE
  L1 Vertical  Accuracy : 98.62%  |  F1: 0.9839
  L2 Subdomain Accuracy : 95.52%  |  F1: 0.9294

Expected honest ranges:
  L1: 88-94%   L2: 78-87%
  If you see 99%+ something is wrong (leakage)

Folders to copy into Streamlit project:
  model_level1/   model_level2/   label_encoders/


## Cell 11 — Inference Test

In [ ]:
def quick_predict(text):
    tok  = DistilBertTokenizerFast.from_pretrained(MODEL_L1_DIR)
    m_l1 = DistilBertForSequenceClassification.from_pretrained(MODEL_L1_DIR).eval()
    m_l2 = DistilBertForSequenceClassification.from_pretrained(MODEL_L2_DIR).eval()
    with open(f'{ENCODER_DIR}/le_level1.pkl','rb') as f: le1 = pickle.load(f)
    with open(f'{ENCODER_DIR}/le_level2.pkl','rb') as f: le2 = pickle.load(f)
    with open(f'{ENCODER_DIR}/taxonomy.json')      as f: tax = json.load(f)

    enc = tok(text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LEN)
    with torch.no_grad():
        p1 = F.softmax(m_l1(**enc).logits, dim=-1)[0].numpy()
        p2 = F.softmax(m_l2(**enc).logits, dim=-1)[0].numpy()

    l1_idx   = int(np.argmax(p1))
    l1_label = le1.inverse_transform([l1_idx])[0]
    l1_conf  = float(p1[l1_idx])
    l2_idx   = int(np.argmax(p2))
    l2_label = le2.inverse_transform([l2_idx])[0]

    valid = tax.get(l1_label, [])
    if l2_label not in valid and valid:
        best = max([i for i,c in enumerate(le2.classes_) if c in valid], key=lambda i: p2[i])
        l2_label = le2.inverse_transform([best])[0]

    status = '✅' if l1_conf >= 0.75 else '⚠️  low confidence'
    print(f'{status} [{l1_label} > {l2_label}]  ({l1_conf:.1%})')
    print(f'   "{text}"')
    print()

# These are the exact cases that FAILED before
# After this fix they should all pass
test_cases = [
    ('Who is the Prime Minister of UK and what are his duties?',   'Politics > UK Politics'),
    ('What are the latest developments in Formula 1 this season?', 'Sports > Formula 1'),
    ('Narendra Modi announced new infrastructure projects.',        'Politics > Indian Politics'),
    ('Virat Kohli scored a century in the Test match.',            'Sports > Cricket'),
    ('OpenAI released a new version of ChatGPT.',                  'Technology > AI'),
    ('The RBI increased interest rates to control inflation.',      'Finance > Banking'),
    ('Hey give me a minute I will be right back.',                 'General'),
    ('What is Bitcoin price today?',                               'Finance > Crypto'),
    ('Who won the FIFA World Cup?',                                'Sports > Football'),
    ('What is happening in Russia Ukraine war?',                   'Politics > Geopolitics'),
    ('How to save money and manage budget?',                       'Finance > Personal Finance'),
    ('What is ChatGPT?',                                           'Technology > AI'),
]

print('='*60)
print('  INFERENCE TESTS  (expected label shown)')
print('='*60)
for text, expected in test_cases:
    quick_predict(text)

  INFERENCE TESTS  (expected label shown)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ [Politics > UK Politics]  (99.0%)
   "Who is the Prime Minister of UK and what are his duties?"



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ [Finance > Personal Finance]  (97.1%)
   "How to save money and manage budget?"



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ [Technology > Artificial Intelligence]  (96.2%)
   "What is ChatGPT?"



## Cell 12 — Download models (Colab only)

In [ ]:
# Run this only on Google Colab to download the trained models
try:
    import shutil
    from google.colab import files
    shutil.make_archive('model_level1',   'zip', 'model_level1')
    shutil.make_archive('model_level2',   'zip', 'model_level2')
    shutil.make_archive('label_encoders', 'zip', 'label_encoders')
    files.download('model_level1.zip')
    files.download('model_level2.zip')
    files.download('label_encoders.zip')
    print('✅ Downloads started')
except:
    print('Not on Colab — find model folders in your working directory')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads started


In [ ]:
import pickle, json
from transformers import DistilBertForSequenceClassification

print("="*50)
print("L1 MODEL CHECK")
print("="*50)
m1 = DistilBertForSequenceClassification.from_pretrained("model_level1")
print(f"L1 model num_labels : {m1.config.num_labels}")
with open("label_encoders/le_level1.pkl", "rb") as f:
    le1 = pickle.load(f)
print(f"L1 encoder classes  : {len(le1.classes_)}")
print(f"L1 classes          : {list(le1.classes_)}")
print(f"L1 match            : {'✅ OK' if m1.config.num_labels == len(le1.classes_) else '❌ MISMATCH'}")

print()
print("="*50)
print("L2 MODEL CHECK")
print("="*50)
m2 = DistilBertForSequenceClassification.from_pretrained("model_level2")
print(f"L2 model num_labels : {m2.config.num_labels}")
with open("label_encoders/le_level2.pkl", "rb") as f:
    le2 = pickle.load(f)
print(f"L2 encoder classes  : {len(le2.classes_)}")
print(f"L2 match            : {'✅ OK' if m2.config.num_labels == len(le2.classes_) else '❌ MISMATCH — L2 model was not saved correctly'}")

print()
print("="*50)
print("WHAT TO DO")
print("="*50)
if m2.config.num_labels != len(le2.classes_):
    print("❌ Your model_level2 folder has the WRONG model in it.")
    print("   Most likely Cell 8 (L2 training) either:")
    print("   1. Did not finish running")
    print("   2. Saved to wrong directory")
    print("   3. Got overwritten by L1 model")
    print()
    print("   SOLUTION → Go back to Colab, open train_final.ipynb")
    print("   Run ONLY Cell 8 again (L2 training)")
    print("   Then run Cell 12 again to download ONLY model_level2.zip")
    print("   Unzip and replace your local model_level2/ folder")

L1 MODEL CHECK


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

L1 model num_labels : 5
L1 encoder classes  : 5
L1 classes          : [np.str_('Finance'), np.str_('General'), np.str_('Politics'), np.str_('Sports'), np.str_('Technology')]
L1 match            : ✅ OK

L2 MODEL CHECK


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

L2 model num_labels : 41
L2 encoder classes  : 41
L2 match            : ✅ OK

WHAT TO DO
